# Sepsis Prediction — Feature Engineering

After EDA we know the dataset is spread across 9 tables linked by `person_id` and timestamped at hourly resolution.  
This notebook merges everything into a single patient-hour level feature matrix and prepares it for modelling.

**Plan**
1. Load all training tables  
2. Build a master hourly time-series per patient  
3. Engineer features: vitals, labs, drugs, procedures, devices  
4. Encode categoricals, handle missingness  
5. Add temporal / lag features  
6. Merge demographics  
7. Attach sepsis label  
8. Save the final feature matrix


## 1. Installs and imports

In [ ]:
!pip install pandas numpy scikit-learn imbalanced-learn tqdm -q

import os
import warnings
import pandas as pd
import numpy as np
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
print("Ready")


## 2. Load raw tables

In [ ]:
# Update this path to wherever your training_data folder lives
DATA_PATH = "/content/drive/MyDrive/sepsis-prediction/training_data"

label_df       = pd.read_csv(os.path.join(DATA_PATH, "SepsisLabel_train.csv"))
demo_df        = pd.read_csv(os.path.join(DATA_PATH, "person_demographics_episode_train.csv"))
lab_df         = pd.read_csv(os.path.join(DATA_PATH, "measurement_lab_train.csv"))
vitals_df      = pd.read_csv(os.path.join(DATA_PATH, "measurement_meds_train.csv"))
obs_df         = pd.read_csv(os.path.join(DATA_PATH, "measurement_observation_train.csv"))
drugs_df       = pd.read_csv(os.path.join(DATA_PATH, "drugsexposure_train.csv"))
proc_df        = pd.read_csv(os.path.join(DATA_PATH, "proceduresoccurrences_train.csv"))
devices_df     = pd.read_csv(os.path.join(DATA_PATH, "devices_train.csv"))
observation_df = pd.read_csv(os.path.join(DATA_PATH, "observation_train.csv"))

print("Loaded all tables")
for name, df in [("label", label_df), ("demo", demo_df), ("labs", lab_df),
                 ("vitals", vitals_df), ("meas_obs", obs_df), ("drugs", drugs_df),
                 ("procedures", proc_df), ("devices", devices_df), ("observation", observation_df)]:
    print(f"  {name:15s}  {df.shape}")


## 3. Parse datetime columns

In [ ]:
label_df["measurement_datetime"]   = pd.to_datetime(label_df["measurement_datetime"])
lab_df["measurement_datetime"]     = pd.to_datetime(lab_df["measurement_datetime"])
vitals_df["measurement_datetime"]  = pd.to_datetime(vitals_df["measurement_datetime"])
obs_df["measurement_datetime"]     = pd.to_datetime(obs_df["measurement_datetime"])
drugs_df["drug_datetime_hourly"]   = pd.to_datetime(drugs_df["drug_datetime_hourly"])
proc_df["procedure_datetime_hourly"] = pd.to_datetime(proc_df["procedure_datetime_hourly"])
devices_df["device_datetime_hourly"] = pd.to_datetime(devices_df["device_datetime_hourly"])
demo_df["visit_start_date"]        = pd.to_datetime(demo_df["visit_start_date"])

print("Datetime parsing done")


## 4. Clean the label table

In [ ]:
# 16 rows have a missing datetime — drop them
label_df = label_df.dropna(subset=["measurement_datetime"]).copy()
label_df["hour"] = label_df["measurement_datetime"].dt.floor("h")

print(f"Label rows: {len(label_df):,}")
print("Sepsis distribution:")
print(label_df["SepsisLabel"].value_counts())
print(f"Imbalance ratio: {label_df['SepsisLabel'].value_counts()[0] / label_df['SepsisLabel'].value_counts()[1]:.1f}:1")


## 5. Build a master (patient × hour) backbone

Every other table will be left-joined onto this.

In [ ]:
backbone = label_df[["person_id", "hour", "SepsisLabel"]].copy()
backbone = backbone.sort_values(["person_id", "hour"]).reset_index(drop=True)

print(f"Backbone shape: {backbone.shape}")
backbone.head(3)


## 6. Lab measurements

Each row is already one patient-hour reading. We floor to hour and pivot.

In [ ]:
lab_df["hour"] = lab_df["measurement_datetime"].dt.floor("h")

# Drop the id columns before aggregating
lab_feat_cols = [c for c in lab_df.columns if c not in
                 ["visit_occurrence_id", "person_id", "measurement_datetime", "hour"]]

# Some patients have >1 lab draw in the same hour — take the mean
lab_agg = (lab_df
           .groupby(["person_id", "hour"])[lab_feat_cols]
           .mean()
           .reset_index())

# Rename for cleaner column names
lab_agg.columns = (["person_id", "hour"] +
                   ["lab_" + c.replace(" ", "_").replace("/", "_").replace("[", "").replace("]", "")[:40]
                    for c in lab_feat_cols])

backbone = backbone.merge(lab_agg, on=["person_id", "hour"], how="left")
print(f"After labs  →  {backbone.shape}")


## 7. Vital signs (measurement_meds)

Contains: BP, HR, temperature, SpO2, RR, FiO2.

In [ ]:
vitals_df["hour"] = vitals_df["measurement_datetime"].dt.floor("h")

vital_feat_cols = [c for c in vitals_df.columns if c not in
                   ["visit_occurrence_id", "person_id", "measurement_datetime", "hour"]]

vitals_agg = (vitals_df
              .groupby(["person_id", "hour"])[vital_feat_cols]
              .mean()
              .reset_index())

vitals_agg.columns = (["person_id", "hour"] +
                      ["vital_" + c.replace(" ", "_")[:40] for c in vital_feat_cols])

backbone = backbone.merge(vitals_agg, on=["person_id", "hour"], how="left")
print(f"After vitals  →  {backbone.shape}")


## 8. Clinical observations (GCS, pupils, pulse pressure)

In [ ]:
obs_df["hour"] = obs_df["measurement_datetime"].dt.floor("h")

numeric_obs = obs_df.select_dtypes(include="number").columns.tolist()
numeric_obs = [c for c in numeric_obs if c not in ["visit_occurrence_id", "person_id"]]

obs_agg = (obs_df
           .groupby(["person_id", "hour"])[numeric_obs]
           .mean()
           .reset_index())

obs_agg.columns = (["person_id", "hour"] +
                   ["obs_" + c.replace(" ", "_")[:40] for c in numeric_obs])

backbone = backbone.merge(obs_agg, on=["person_id", "hour"], how="left")
print(f"After obs measurements  →  {backbone.shape}")


## 9. Drug exposure — binary flags per hour

Vasoactive drugs (epinephrine, norepinephrine, dopamine, milrinone) and key antibiotics  
are strong clinical signals for severity. We create binary flags: was this drug given in this hour?


In [ ]:
drugs_df["hour"] = drugs_df["drug_datetime_hourly"].dt.floor("h")

# Keep top drugs (covers ~90%+ of volume)
top_drugs = drugs_df["drug_concept_id"].value_counts().head(20).index.tolist()

drug_pivot = (drugs_df[drugs_df["drug_concept_id"].isin(top_drugs)]
              .assign(flag=1)
              .pivot_table(index=["person_id", "hour"],
                           columns="drug_concept_id",
                           values="flag",
                           aggfunc="max",
                           fill_value=0)
              .reset_index())

drug_pivot.columns = (["person_id", "hour"] +
                      ["drug_" + str(c).replace(" ", "_") for c in drug_pivot.columns[2:]])

backbone = backbone.merge(drug_pivot, on=["person_id", "hour"], how="left")

# Fill NaN drug flags with 0 (not given)
drug_cols = [c for c in backbone.columns if c.startswith("drug_")]
backbone[drug_cols] = backbone[drug_cols].fillna(0).astype(int)

print(f"After drugs  →  {backbone.shape}")
print(f"Drug features: {len(drug_cols)}")


## 10. Procedure flags

Invasive ventilation, dialysis, ECMO are strong severity markers.

In [ ]:
proc_df["hour"] = proc_df["procedure_datetime_hourly"].dt.floor("h")

top_procs = proc_df["procedure"].value_counts().head(15).index.tolist()

proc_pivot = (proc_df[proc_df["procedure"].isin(top_procs)]
              .assign(flag=1)
              .pivot_table(index=["person_id", "hour"],
                           columns="procedure",
                           values="flag",
                           aggfunc="max",
                           fill_value=0)
              .reset_index())

proc_pivot.columns = (["person_id", "hour"] +
                      ["proc_" + c.replace(" ", "_") for c in proc_pivot.columns[2:]])

backbone = backbone.merge(proc_pivot, on=["person_id", "hour"], how="left")

proc_cols = [c for c in backbone.columns if c.startswith("proc_")]
backbone[proc_cols] = backbone[proc_cols].fillna(0).astype(int)

print(f"After procedures  →  {backbone.shape}")


## 11. Device flags

Presence of an arterial line or central venous catheter indicates ICU severity.

In [ ]:
devices_df["hour"] = devices_df["device_datetime_hourly"].dt.floor("h")

top_devices = devices_df["device"].value_counts().head(10).index.tolist()

dev_pivot = (devices_df[devices_df["device"].isin(top_devices)]
             .assign(flag=1)
             .pivot_table(index=["person_id", "hour"],
                          columns="device",
                          values="flag",
                          aggfunc="max",
                          fill_value=0)
             .reset_index())

dev_pivot.columns = (["person_id", "hour"] +
                     ["dev_" + c.replace(" ", "_") for c in dev_pivot.columns[2:]])

backbone = backbone.merge(dev_pivot, on=["person_id", "hour"], how="left")

dev_cols = [c for c in backbone.columns if c.startswith("dev_")]
backbone[dev_cols] = backbone[dev_cols].fillna(0).astype(int)

print(f"After devices  →  {backbone.shape}")


## 12. Patient demographics

Age and gender from the demographics table.

In [ ]:
# Keep one row per patient (demographics don't change)
demo_patient = demo_df[["person_id", "age_in_months", "gender"]].drop_duplicates("person_id")

# Encode gender
demo_patient["gender_male"] = (demo_patient["gender"] == "MALE").astype(int)
demo_patient = demo_patient.drop(columns=["gender"])

backbone = backbone.merge(demo_patient, on="person_id", how="left")
print(f"After demographics  →  {backbone.shape}")


## 13. Temporal features

How many hours has the patient been in the ICU? Day of week and hour of day can also matter.

In [ ]:
# Hours since ICU admission (time since first observed hour for each patient)
backbone = backbone.sort_values(["person_id", "hour"])
backbone["icu_hour"] = backbone.groupby("person_id")["hour"].transform(
    lambda x: (x - x.min()).dt.total_seconds() / 3600
).astype(int)

backbone["hour_of_day"] = backbone["hour"].dt.hour
backbone["day_of_week"]  = backbone["hour"].dt.dayofweek

print("Temporal features added")
backbone[["person_id", "hour", "icu_hour", "hour_of_day", "day_of_week"]].head(5)


## 14. Lag and rolling features

For the most clinically important vitals (HR, SBP, SpO2, temperature, RR) we add:
- **Lag 1 / 2 / 6** — value at previous hours  
- **Rolling mean & std** over 3 h and 6 h windows  
- **Delta** — change from previous hour

This gives the model a memory of recent trend without needing a sequence model at this stage.


In [ ]:
vital_trend_cols = [c for c in backbone.columns if c.startswith("vital_")]

backbone = backbone.sort_values(["person_id", "hour"])

for col in tqdm(vital_trend_cols, desc="Lag & rolling features"):
    grp = backbone.groupby("person_id")[col]

    backbone[col + "_lag1"] = grp.shift(1)
    backbone[col + "_lag6"] = grp.shift(6)
    backbone[col + "_delta1"] = backbone[col] - grp.shift(1)
    backbone[col + "_roll3_mean"] = grp.transform(lambda x: x.rolling(3, min_periods=1).mean())
    backbone[col + "_roll6_std"]  = grp.transform(lambda x: x.rolling(6, min_periods=1).std())

print(f"Shape after lag/rolling:  {backbone.shape}")


## 15. Missing value summary

In [ ]:
miss = backbone.isnull().sum()
miss_pct = (miss / len(backbone) * 100).round(1)
miss_report = pd.DataFrame({"missing_n": miss, "missing_pct": miss_pct})
miss_report = miss_report[miss_report["missing_n"] > 0].sort_values("missing_pct", ascending=False)

print(f"Columns with missing values: {len(miss_report)}")
print(miss_report.head(20).to_string())


## 16. Imputation strategy

Labs and vitals are **not missing at random** — they're only recorded when a test is ordered.  
We use **forward-fill within each patient** (carry last known value), then fill remaining  
NaNs with the global median. Binary flags (drug/procedure/device) are already 0 for absence.


In [ ]:
backbone = backbone.sort_values(["person_id", "hour"])

# Columns to impute (not IDs, label, or binary flags)
continuous_cols = [c for c in backbone.columns if c not in
                   ["person_id", "hour", "SepsisLabel", "icu_hour", "hour_of_day",
                    "day_of_week", "gender_male"] +
                   drug_cols + proc_cols + dev_cols]

# Forward fill within patient
backbone[continuous_cols] = (backbone
                             .groupby("person_id")[continuous_cols]
                             .transform(lambda x: x.ffill()))

# Fill residual NaN with column median
for col in continuous_cols:
    backbone[col] = backbone[col].fillna(backbone[col].median())

print(f"Missing values remaining: {backbone.isnull().sum().sum()}")


## 17. Drop near-constant or all-zero columns

In [ ]:
# Drop columns that are >98% zero (uninformative binary flags)
binary_cols = drug_cols + proc_cols + dev_cols
zero_frac = backbone[binary_cols].mean()
drop_binary = zero_frac[zero_frac < 0.02].index.tolist()

backbone = backbone.drop(columns=drop_binary)
print(f"Dropped {len(drop_binary)} near-zero binary columns")
print(f"Final shape: {backbone.shape}")


## 18. Sanity check — final dataset

In [ ]:
print("Shape:", backbone.shape)
print("\nLabel distribution:")
print(backbone["SepsisLabel"].value_counts())
print(f"\nImbalance ratio: {backbone['SepsisLabel'].value_counts()[0]/backbone['SepsisLabel'].value_counts()[1]:.1f}:1")
print(f"\nUnique patients: {backbone['person_id'].nunique()}")
print(f"Missing values: {backbone.isnull().sum().sum()}")
backbone.head(3)


## 19. Save the feature matrix

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/sepsis-prediction"

backbone.to_parquet(os.path.join(SAVE_PATH, "features_train.parquet"), index=False)

print("Saved  →  features_train.parquet")
print(f"Final shape: {backbone.shape}")


---
## What's next

| Notebook | What it covers |
|---|---|
| `02_xgboost_baseline.ipynb` | XGBoost + SMOTE, AUROC, F1, fold-wise OOF |
| `03_shap_analysis.ipynb` | SHAP values, feature importance, waterfall plots |
| `04_lstm.ipynb` | Sequence model on the hourly time-series |
| `05_tft.ipynb` | Temporal Fusion Transformer, multi-horizon prediction |
